# Agency Context -> Plan -> Prompt Starter Notebook

This notebook builds the first shippable layer of your system:

1. agency owner provides messy context
2. system deconstructs it into a strict intermediate representation
3. system builds a plan
4. system generates an execution prompt bundle

Strong opinion:
- Do **not** start with subagents
- Do **not** start with web research
- Do **not** let the model jump directly from raw context to action

Treat this as a compiler pipeline.

## What this notebook does

This version is intentionally narrow and practical.

It gives you:
- a structured `ContextIR`
- deterministic extraction for obvious signals
- heuristic task classification
- a plan builder
- a prompt builder
- confidence / ambiguity flags

Later, you can replace the extraction or planning step with an LLM. But first make the data contract stable.

In [ ]:
from __future__ import annotations

import json
import re
import hashlib
from dataclasses import dataclass, asdict, field
from datetime import datetime, timezone
from typing import List, Dict, Any, Optional
from pprint import pprint

In [ ]:
@dataclass
class Evidence:
    field_name: str
    value: Any
    source_text: str
    confidence: float


@dataclass
class ContextIR:
    context_id: str
    created_at_utc: str
    raw_context: str

    business_name: Optional[str] = None
    business_type: Optional[str] = None
    core_offer: List[str] = field(default_factory=list)
    target_audience: List[str] = field(default_factory=list)
    goals: List[str] = field(default_factory=list)
    deliverables: List[str] = field(default_factory=list)
    constraints: List[str] = field(default_factory=list)
    exclusions: List[str] = field(default_factory=list)
    channels: List[str] = field(default_factory=list)
    tools: List[str] = field(default_factory=list)
    tone: List[str] = field(default_factory=list)
    assets: List[str] = field(default_factory=list)
    urls: List[str] = field(default_factory=list)
    budget_notes: List[str] = field(default_factory=list)
    deadlines: List[str] = field(default_factory=list)
    open_questions: List[str] = field(default_factory=list)
    assumptions: List[str] = field(default_factory=list)
    task_type: Optional[str] = None
    confidence_score: float = 0.0
    evidence: List[Dict[str, Any]] = field(default_factory=list)


@dataclass
class ExecutionPlan:
    plan_name: str
    objective: str
    ordered_steps: List[str]
    risks: List[str]
    required_inputs: List[str]
    output_format: str
    stop_conditions: List[str]


@dataclass
class PromptBundle:
    system_prompt: str
    user_prompt: str
    planner_prompt: str
    missing_info_prompt: str

## Input

Put the agency owner's raw context below.

This should be whatever they actually dump into the system: notes, bullets, links, offer description, audience, preferences, deadlines, and anything else.

In [ ]:
raw_context = '''
Agency name: BrightLoop Media
We help SaaS founders with LinkedIn content, ghostwriting, and outbound email strategy.
Ideal clients are B2B SaaS founders doing between $10k MRR and $200k MRR.
We do not want ecommerce or local business clients.
Tone should be sharp, credible, not cringe, not hypey.
Primary goal right now is get 10 qualified discovery calls in the next 45 days.
Founder has case studies but they are messy notes, loom recordings, and screenshots.
Budget is lean, so prefer organic, founder-led distribution over paid ads.
Main channels: LinkedIn, cold email, website landing page.
Need agent to create a practical GTM plan and then write prompts for outreach assets.
Avoid generic marketing fluff. Prefer direct language and clear CTAs.
We use Notion, Google Drive, Apollo, Clay, and ChatGPT.
Website: https://brightloop.example
'''.strip()

print(raw_context)

In [ ]:
SECTION_PATTERNS = {
    "goal": [r"\bgoal\b", r"\bobjective\b", r"\bwant\b", r"\bneed\b"],
    "constraint": [r"\bconstraint\b", r"\bmust\b", r"\bavoid\b", r"\bdo not\b", r"\bdon't\b", r"\bprefer\b"],
    "deliverable": [r"\bdeliverable\b", r"\boutput\b", r"\bcreate\b", r"\bwrite\b", r"\bbuild\b", r"\bplan\b"],
    "audience": [r"\bideal client\b", r"\baudience\b", r"\bICP\b", r"\bfor\b"],
    "tool": [r"\buse\b", r"\btools\b", r"\bstack\b"],
}

TASK_TYPE_KEYWORDS = {
    "gtm_strategy": ["gtm", "go to market", "launch", "distribution", "positioning"],
    "outreach_system": ["cold email", "outbound", "lead list", "apollo", "clay", "discovery call"],
    "content_strategy": ["linkedin", "ghostwriting", "content calendar", "posts"],
    "offer_packaging": ["pricing", "offer", "package", "tier"],
    "landing_page": ["landing page", "website copy", "homepage"],
}

TOOL_CANDIDATES = [
    "Notion", "Google Drive", "Apollo", "Clay", "ChatGPT", "HubSpot", "Slack", "Sheets", "Google Docs", "Airtable"
]

CHANNEL_CANDIDATES = [
    "LinkedIn", "cold email", "website", "YouTube", "X", "Twitter", "newsletter", "Instagram", "Facebook"
]

In [ ]:
def normalize_whitespace(text: str) -> str:
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def extract_urls(text: str) -> List[str]:
    return sorted(set(re.findall(r"https?://[^\s)]+", text)))


def extract_business_name(text: str) -> Optional[str]:
    patterns = [
        r"Agency name:\s*(.+)",
        r"Business name:\s*(.+)",
        r"Company name:\s*(.+)",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return None


def extract_tools(text: str) -> List[str]:
    found = []
    lowered = text.lower()
    for tool in TOOL_CANDIDATES:
        if tool.lower() in lowered:
            found.append(tool)
    return sorted(set(found))


def extract_channels(text: str) -> List[str]:
    found = []
    lowered = text.lower()
    for channel in CHANNEL_CANDIDATES:
        if channel.lower() in lowered:
            found.append(channel)
    return sorted(set(found))


def split_statements(text: str) -> List[str]:
    rough = []
    for line in text.split("\n"):
        line = line.strip(" -•\t")
        if not line:
            continue
        parts = re.split(r"(?<=[.!?])\s+", line)
        rough.extend([p.strip() for p in parts if p.strip()])
    return rough


def classify_task_type(text: str) -> str:
    lowered = text.lower()
    scores = {}
    for task_type, keywords in TASK_TYPE_KEYWORDS.items():
        score = sum(1 for kw in keywords if kw in lowered)
        if score:
            scores[task_type] = score
    if not scores:
        return "general_strategy"
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)[0][0]


def infer_business_type(text: str) -> Optional[str]:
    lowered = text.lower()
    if "agency" in lowered:
        return "agency"
    if "saas" in lowered:
        return "saas"
    if "consulting" in lowered:
        return "consulting"
    return None


def score_confidence(ir: ContextIR) -> float:
    score = 0.0
    if ir.business_name:
        score += 0.1
    if ir.core_offer:
        score += 0.15
    if ir.target_audience:
        score += 0.15
    if ir.goals:
        score += 0.15
    if ir.deliverables:
        score += 0.15
    if ir.constraints or ir.exclusions:
        score += 0.1
    if ir.channels:
        score += 0.05
    if ir.tools:
        score += 0.05
    if ir.tone:
        score += 0.05
    if ir.task_type:
        score += 0.05
    return round(min(score, 1.0), 2)

In [ ]:
def parse_context(raw_text: str) -> ContextIR:
    text = normalize_whitespace(raw_text)
    statements = split_statements(text)
    now = datetime.now(timezone.utc).isoformat()
    context_id = hashlib.md5(text.encode("utf-8")).hexdigest()[:12]

    ir = ContextIR(
        context_id=context_id,
        created_at_utc=now,
        raw_context=text,
        business_name=extract_business_name(text),
        business_type=infer_business_type(text),
        urls=extract_urls(text),
        tools=extract_tools(text),
        channels=extract_channels(text),
        task_type=classify_task_type(text),
    )

    for statement in statements:
        s = statement.strip()
        s_lower = s.lower()

        if "we help" in s_lower or "we do" in s_lower:
            ir.core_offer.append(s)
            ir.evidence.append(asdict(Evidence("core_offer", s, s, 0.85)))

        if any(k in s_lower for k in ["ideal client", "ideal clients", "icp", "audience"]):
            ir.target_audience.append(s)
            ir.evidence.append(asdict(Evidence("target_audience", s, s, 0.9)))

        if any(k in s_lower for k in ["goal", "primary goal", "objective", "north star"]):
            ir.goals.append(s)
            ir.evidence.append(asdict(Evidence("goals", s, s, 0.9)))

        if any(k in s_lower for k in ["need agent to", "need to", "create", "write", "build a", "build an", "gtm plan", "prompt"]):
            ir.deliverables.append(s)
            ir.evidence.append(asdict(Evidence("deliverables", s, s, 0.8)))

        if any(k in s_lower for k in ["do not", "don't", "avoid", "must not"]):
            ir.exclusions.append(s)
            ir.evidence.append(asdict(Evidence("exclusions", s, s, 0.9)))

        if any(k in s_lower for k in ["prefer", "budget", "lean", "constraint", "must"]):
            ir.constraints.append(s)
            ir.evidence.append(asdict(Evidence("constraints", s, s, 0.75)))

        if any(k in s_lower for k in ["tone", "voice", "style"]):
            ir.tone.append(s)
            ir.evidence.append(asdict(Evidence("tone", s, s, 0.85)))

        if any(k in s_lower for k in ["case studies", "loom", "screenshots", "assets", "docs", "notes"]):
            ir.assets.append(s)
            ir.evidence.append(asdict(Evidence("assets", s, s, 0.75)))

        if re.search(r"\b\d+\s*(days|weeks|months)\b", s_lower):
            ir.deadlines.append(s)
            ir.evidence.append(asdict(Evidence("deadlines", s, s, 0.8)))

        if "$" in s or "budget" in s_lower:
            ir.budget_notes.append(s)
            ir.evidence.append(asdict(Evidence("budget_notes", s, s, 0.8)))

    dedupe_fields = [
        "core_offer", "target_audience", "goals", "deliverables", "constraints",
        "exclusions", "channels", "tools", "tone", "assets", "urls",
        "budget_notes", "deadlines"
    ]

    for field_name in dedupe_fields:
        values = getattr(ir, field_name)
        setattr(ir, field_name, list(dict.fromkeys(values)))

    if not ir.goals:
        ir.open_questions.append("What is the primary measurable goal for this task?")
    if not ir.target_audience:
        ir.open_questions.append("Who is the target audience or ICP?")
    if not ir.deliverables:
        ir.open_questions.append("What exact deliverable should the agent produce first?")
    if not ir.tone:
        ir.assumptions.append("Default to concise, credible, direct business language.")
    if not ir.channels:
        ir.assumptions.append("Assume web and text-based channels unless specified otherwise.")

    ir.confidence_score = score_confidence(ir)
    return ir

In [ ]:
ir = parse_context(raw_context)
pprint(asdict(ir))

## Plan builder

The plan should be derived from the structured context, not from raw text.

This matters because you want:
- predictable planning
- inspectable logic
- easier debugging when the agent goes wrong

In [ ]:
def build_plan(ir: ContextIR) -> ExecutionPlan:
    objective = ir.goals[0] if ir.goals else "Create the first useful deliverable from the provided business context."

    common_risks = [
        "Vague goal leads to generic output.",
        "Missing audience detail causes weak positioning.",
        "Unclear exclusions make the plan over-broad.",
        "Messy proof assets reduce credibility unless normalized first.",
    ]

    if ir.task_type == "gtm_strategy":
        steps = [
            "Normalize the business context into a single source-of-truth brief.",
            "Extract offer, ICP, goal, constraints, channels, proof assets, and tone.",
            "Define the narrowest viable GTM objective tied to a measurable target.",
            "Choose 1-2 primary channels based on speed, budget, and founder fit.",
            "Map the funnel: audience -> message -> asset -> CTA -> conversion event.",
            "Turn messy proof assets into reusable evidence blocks.",
            "Generate the first execution assets required for launch.",
        ]
        output_format = "A structured GTM brief followed by a 30-45 day action plan and asset prompts."
        required_inputs = [
            "Business context",
            "Goal metric",
            "Audience / ICP",
            "Offer details",
            "Channel preferences",
            "Available proof assets",
        ]
    elif ir.task_type == "outreach_system":
        steps = [
            "Define ICP and exclude non-fit segments.",
            "Translate the offer into one sharp problem-solution angle.",
            "Design the lead selection logic and qualification filters.",
            "Build outreach structure: opener, relevance, credibility, CTA.",
            "Create variants for different audience slices.",
            "Define reply handling rules and next-step prompts.",
        ]
        output_format = "An outreach playbook with qualification rules, messaging structure, and prompt templates."
        required_inputs = [
            "ICP",
            "Offer",
            "Proof assets",
            "Exclusions",
            "Channel",
        ]
    elif ir.task_type == "content_strategy":
        steps = [
            "Clarify the audience and the business outcome content must drive.",
            "Extract founder positioning, proof, and repeated customer pain points.",
            "Define content pillars tied to offer demand, not vanity posting.",
            "Create post archetypes and CTA logic.",
            "Build a publishing plan and prompt bundle.",
        ]
        output_format = "A content strategy brief with pillars, post types, and prompt templates."
        required_inputs = [
            "Audience",
            "Offer",
            "Business goal",
            "Tone",
            "Proof assets",
        ]
    else:
        steps = [
            "Convert raw context into a normalized business brief.",
            "Identify the highest-value first deliverable.",
            "List assumptions, gaps, and non-negotiable constraints.",
            "Create the minimum viable execution plan.",
            "Generate a reusable prompt bundle.",
        ]
        output_format = "A structured brief, execution plan, and reusable prompts."
        required_inputs = ["Business context", "Goal", "Constraints", "Desired deliverable"]

    stop_conditions = [
        "Primary goal is not measurable.",
        "Target audience is missing or contradictory.",
        "Deliverable is still ambiguous after normalization.",
    ]

    return ExecutionPlan(
        plan_name=f"{ir.task_type}_plan",
        objective=objective,
        ordered_steps=steps,
        risks=common_risks,
        required_inputs=required_inputs,
        output_format=output_format,
        stop_conditions=stop_conditions,
    )

In [ ]:
plan = build_plan(ir)
pprint(asdict(plan))

## Prompt builder

You need at least three prompt layers:

1. **system prompt** for role, rules, and failure boundaries
2. **planner prompt** for reasoning over the structured brief
3. **user prompt** for the actual task execution

Also generate a missing-info prompt so the system can ask precise follow-ups instead of vague questions.

In [ ]:
def bullet_join(items: List[str], default: str = "None provided") -> str:
    clean = [x for x in items if x and x.strip()]
    if not clean:
        return f"- {default}"
    return "\n".join(f"- {x}" for x in clean)


def build_prompt_bundle(ir: ContextIR, plan: ExecutionPlan) -> PromptBundle:
    system_prompt = f"""
You are a business-context compiler and execution planner.

Rules:
1. Do not invent facts that are not present in the structured brief.
2. Prefer concrete language over generic strategy language.
3. Respect exclusions and constraints as hard boundaries.
4. If important information is missing, surface the gap explicitly.
5. Produce outputs that are directly usable by an operator or downstream agent.
6. Keep recommendations tied to the stated business goal.
""".strip()

    planner_prompt = f"""
Use the structured context below to create a high-signal plan.

Business name: {ir.business_name or 'Unknown'}
Business type: {ir.business_type or 'Unknown'}
Task type: {ir.task_type or 'Unknown'}

Core offer:
{bullet_join(ir.core_offer)}

Target audience:
{bullet_join(ir.target_audience)}

Goals:
{bullet_join(ir.goals)}

Deliverables:
{bullet_join(ir.deliverables)}

Constraints:
{bullet_join(ir.constraints)}

Exclusions:
{bullet_join(ir.exclusions)}

Channels:
{bullet_join(ir.channels)}

Tools:
{bullet_join(ir.tools)}

Tone:
{bullet_join(ir.tone)}

Assets / proof:
{bullet_join(ir.assets)}

Your job:
- restate the core objective in one sentence
- identify the narrowest practical first move
- produce an ordered plan
- list assumptions and unresolved gaps
- do not broaden scope beyond the provided context
""".strip()

    user_prompt = f"""
Create the requested output using the plan below.

Objective:
{plan.objective}

Ordered steps:
{bullet_join(plan.ordered_steps)}

Required inputs:
{bullet_join(plan.required_inputs)}

Risks:
{bullet_join(plan.risks)}

Output format:
- {plan.output_format}

Guardrails:
- use direct, practical language
- avoid filler and marketing clichés
- keep the output grounded in the given audience, channels, and goal
- make assumptions explicit
""".strip()

    missing_info_prompt = f"""
Ask only the minimum missing questions needed to improve the next output.

Known gaps:
{bullet_join(ir.open_questions + ir.assumptions, default='No major gaps detected')}

Return:
- at most 5 questions
- each question must be specific
- each question must explain why it matters
""".strip()

    return PromptBundle(
        system_prompt=system_prompt,
        user_prompt=user_prompt,
        planner_prompt=planner_prompt,
        missing_info_prompt=missing_info_prompt,
    )

In [ ]:
prompt_bundle = build_prompt_bundle(ir, plan)
pprint(asdict(prompt_bundle))

## Human-readable render

This is the exact shape I would inspect in the notebook before adding more complexity.

In [ ]:
def render_summary(ir: ContextIR, plan: ExecutionPlan, pb: PromptBundle) -> str:
    return f"""
# Context Summary

## Business
- Name: {ir.business_name or 'Unknown'}
- Type: {ir.business_type or 'Unknown'}
- Task Type: {ir.task_type or 'Unknown'}
- Confidence: {ir.confidence_score}

## Offer
{bullet_join(ir.core_offer)}

## Audience
{bullet_join(ir.target_audience)}

## Goal
{bullet_join(ir.goals)}

## Constraints
{bullet_join(ir.constraints)}

## Exclusions
{bullet_join(ir.exclusions)}

## Deliverables
{bullet_join(ir.deliverables)}

## Channels
{bullet_join(ir.channels)}

## Plan
{bullet_join(plan.ordered_steps)}

## Open Questions
{bullet_join(ir.open_questions, default='No open questions')}
""".strip()

summary_text = render_summary(ir, plan, prompt_bundle)
print(summary_text)

## Optional next layer: LLM extraction

Do this only after the deterministic version feels stable.

The LLM should not receive only the raw context.
It should receive:
- the raw context
- the deterministic extraction
- a strict schema to fill
- instructions to preserve uncertainty

That way the model refines, rather than hallucinates from scratch.

In [ ]:
LLM_EXTRACTION_PROMPT = """
You are given raw business context and a baseline deterministic extraction.
Your job is to improve the extraction without inventing facts.

Return valid JSON only.

Rules:
1. Keep uncertainty explicit.
2. Do not infer a fact unless there is textual evidence.
3. If a field is missing, leave it empty.
4. Preserve exclusions and constraints exactly.
5. Keep deliverables concrete and minimal.
""".strip()

print(LLM_EXTRACTION_PROMPT)

## Evaluation

Before routing to any downstream agent, validate three things:

1. Did we extract the real goal, or just restate fluff?
2. Are the exclusions and constraints preserved?
3. Does the first plan step reduce uncertainty or create more of it?

If this layer fails, downstream agents will execute garbage with confidence.

In [ ]:
def validate_ir(ir: ContextIR) -> Dict[str, Any]:
    issues = []

    if not ir.goals:
        issues.append("Missing measurable goal.")
    if not ir.target_audience:
        issues.append("Missing target audience / ICP.")
    if not ir.deliverables:
        issues.append("Missing explicit deliverable.")
    if ir.confidence_score < 0.6:
        issues.append("Low confidence extraction. Ask follow-up questions before execution.")
    if len(ir.constraints) == 0 and len(ir.exclusions) == 0:
        issues.append("No hard boundaries found. Risk of generic or over-broad output.")

    return {
        "ready_for_downstream_agent": len(issues) == 0,
        "issues": issues,
    }

validation = validate_ir(ir)
pprint(validation)

## What I would build next

After this notebook works on 10-20 real contexts, add the next layer in this order:

1. schema-enforced LLM refinement
2. task-type routing
3. gap-triggered web research for missing external facts
4. prompt optimization per task type
5. only then consider subagents

Do not reverse that order.